# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 37.6MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=27215d12-af65-42d5-ac04-4991fd58f505
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 65.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 131MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [5]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    if rand_init:
        random_values = torch.rand_like(x)
        delta = (random_values * 2 - 1) * eps
    else:
        delta = torch.zeros_like(x)

    delta = clip_box(x, delta)

    for step in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)

    x_adv = (x + delta).detach()
    return x_adv


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    if rand_init:
        direction = torch.randn_like(x)
        direction = direction / (l2_norm(direction) + 1e-12)
        radius = torch.rand(x.shape[0], 1, 1, 1, device=x.device) * eps
        delta = direction * radius
    else:
        delta = torch.zeros_like(x)

    delta = clip_box(x, delta)

    for step in range(steps):
        grad = input_grad(model, x, delta, y)
        grad_norm = l2_norm(grad)
        delta = delta + alpha * grad / (grad_norm + 1e-12)

        delta_norm = l2_norm(delta)
        scale = eps / delta_norm.clamp(min=eps)
        delta = delta * scale

        delta = clip_box(x, delta)

    x_adv = (x + delta).detach()
    return x_adv

In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [ ]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

KeyboardInterrupt: 

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [8]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [9]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    logits = model(x_adv)
    loss = F.cross_entropy(logits, y)
    return loss

In [ ]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

In [ ]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

In [ ]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

In [10]:
def standard_loss(model, x, y):
    model.train()
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    return loss

In [ ]:
model_std = adv_train(make_model(), standard_loss, epochs=5)
model_adv = adv_train(make_model(), single_norm_loss, epochs=5, norm="linf")

In [11]:
def eval_linf(model, eps_255, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)
    eps = eps_255 / 255
    alpha = 2.5 * eps / steps
    clean_correct = 0
    robust_correct = 0
    for i in range(0, len(x_test), batch_size):
        x = x_test[i:i + batch_size]
        y = y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        robust = clean.clone()
        for _ in range(restarts):
            x_adv = pgd_linf(model, x, y, eps, alpha, steps)
            with torch.no_grad():
                robust = robust & (model(x_adv).argmax(1) == y)
        clean_correct += clean.sum().item()
        robust_correct += robust.sum().item()
    clean_acc = 100.0 * clean_correct / len(x_test)
    robust_acc = 100.0 * robust_correct / len(x_test)
    return clean_acc, robust_acc

In [15]:
eps_list = [0, 1, 2, 4, 8, 16]

print("Standard model")
for eps_255 in eps_list:
    clean_acc, robust_acc = eval_linf(model_std, eps_255)
    print("eps =", eps_255, "/255   clean", clean_acc, "  robust", robust_acc)

print("Adversarially trained model")
for eps_255 in eps_list:
    clean_acc, robust_acc = eval_linf(model_adv, eps_255)
    print("eps =", eps_255, "/255   clean", clean_acc, "  robust", robust_acc)

Standard model


NameError: name 'model_std' is not defined

In [12]:
def eval_fgsm(model, eps_255, batch_size=250):
    model.eval()
    eps = eps_255 / 255
    clean_correct = 0
    robust_correct = 0
    for i in range(0, len(x_test), batch_size):
        x = x_test[i:i + batch_size]
        y = y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        x_adv = pgd_linf(model, x, y, eps, eps, 1, rand_init=False)
        with torch.no_grad():
            robust = clean & (model(x_adv).argmax(1) == y)
        clean_correct += clean.sum().item()
        robust_correct += robust.sum().item()
    clean_acc = 100.0 * clean_correct / len(x_test)
    robust_acc = 100.0 * robust_correct / len(x_test)
    return clean_acc, robust_acc

In [ ]:
eps_list = [0, 1, 2, 4, 8, 16]

print("Standard model, FGSM")
for eps_255 in eps_list:
    clean_acc, fgsm_acc = eval_fgsm(model_std, eps_255)
    print("eps =", eps_255, "/255   clean", clean_acc, "  fgsm", fgsm_acc)

print("Adversarially trained model, FGSM")
for eps_255 in eps_list:
    clean_acc, fgsm_acc = eval_fgsm(model_adv, eps_255)
    print("eps =", eps_255, "/255   clean", clean_acc, "  fgsm", fgsm_acc)

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [13]:
def ramp_loss(model, x, y, lam=0.5):

    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()

    logits_q = model(x_q)
    logits_r = model(x_r)

    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    ce_max = torch.maximum(ce_q, ce_r)

    log_p_q = F.log_softmax(logits_q, dim=1)
    log_p_r = F.log_softmax(logits_r, dim=1)
    p_q = log_p_q.exp()
    kl_per_image = (p_q * (log_p_q - log_p_r)).sum(dim=1)

    correct_q = logits_q.argmax(dim=1) == y
    if correct_q.any():
        kl = kl_per_image[correct_q].mean()
    else:
        kl = torch.zeros((), device=x.device)

    loss = ce_max.mean() + lam * kl
    return loss



def gradient_projection(g_n, g_a):
    dot = (g_n * g_a).sum()
    norms = g_n.norm() * g_a.norm() + 1e-12
    cos = dot / norms
    if cos > 0:
        return cos * g_n
    else:
        return torch.zeros_like(g_n)

In [11]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.434 (68 s)
epoch 2: loss 1.337 (130 s)
epoch 3: loss 1.297 (192 s)
epoch 4: loss 1.274 (254 s)
epoch 5: loss 1.247 (315 s)
epoch 6: loss 1.223 (377 s)
epoch 7: loss 1.201 (439 s)
epoch 8: loss 1.184 (500 s)
epoch 9: loss 1.174 (562 s)
epoch 10: loss 1.165 (624 s)
clean 79.0  linf 42.5  l2 62.0  l1 45.2  union 39.8   (73 s)


{'clean': 79.0, 'linf': 42.5, 'l2': 62.0, 'l1': 45.2, 'union': 39.8}

In [19]:
base_results = test_model(load_base_model())

clean 78.7  linf 48.4  l2 59.8  l1 22.3  union 22.2   (67 s)


# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [14]:

ref_model = load_base_model()
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)


def my_loss(model, x, y, lam=0.5, gamma=1.0):

    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()

    logits_q = model(x_q)
    logits_r = model(x_r)

    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    ce_max = torch.maximum(ce_q, ce_r)


    log_p_q = F.log_softmax(logits_q, dim=1)
    log_p_r = F.log_softmax(logits_r, dim=1)
    p_q = log_p_q.exp()
    kl_per_image = (p_q * (log_p_q - log_p_r)).sum(dim=1)
    correct_q = logits_q.argmax(dim=1) == y
    if correct_q.any():
        kl = kl_per_image[correct_q].mean()
    else:
        kl = torch.zeros((), device=x.device)

    with torch.no_grad():
        p_ref = F.softmax(ref_model(x_q), dim=1)
    anchor = F.kl_div(log_p_q, p_ref, reduction="batchmean")

    loss = ce_max.mean() + lam * kl + gamma * anchor
    return loss

In [24]:
model_check = adv_train(load_base_model(), my_loss, epochs=1)


epoch 1: loss 1.474 (56 s)


In [25]:
model_mine_s0 = adv_train(load_base_model(), my_loss, seed=0)
res_mine_s0 = test_model(model_mine_s0)

epoch 1: loss 1.488 (54 s)
epoch 2: loss 1.379 (108 s)
epoch 3: loss 1.328 (163 s)
epoch 4: loss 1.296 (218 s)
epoch 5: loss 1.264 (273 s)
epoch 6: loss 1.228 (328 s)
epoch 7: loss 1.201 (384 s)
epoch 8: loss 1.184 (439 s)
epoch 9: loss 1.172 (495 s)
epoch 10: loss 1.163 (551 s)
clean 77.6  linf 44.8  l2 62.0  l1 47.6  union 42.5   (69 s)


In [17]:
model_ramp_s1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1)
res_ramp_s1 = test_model(model_ramp_s1)

epoch 1: loss 1.437 (72 s)


KeyboardInterrupt: 

In [29]:
model_mine_s1 = adv_train(load_base_model(), my_loss, seed=1)
res_mine_s1 = test_model(model_mine_s1)

epoch 1: loss 1.501 (56 s)
epoch 2: loss 1.383 (111 s)
epoch 3: loss 1.335 (167 s)
epoch 4: loss 1.296 (222 s)
epoch 5: loss 1.260 (278 s)
epoch 6: loss 1.232 (333 s)
epoch 7: loss 1.199 (389 s)
epoch 8: loss 1.184 (445 s)
epoch 9: loss 1.166 (500 s)
epoch 10: loss 1.168 (556 s)
clean 76.2  linf 45.1  l2 61.8  l1 48.9  union 42.9   (69 s)


In [30]:
model_ablate = adv_train(load_base_model(), my_loss, seed=0, gamma=0.0)
res_ablate = test_model(model_ablate)

epoch 1: loss 1.422 (55 s)
epoch 2: loss 1.318 (111 s)
epoch 3: loss 1.275 (167 s)
epoch 4: loss 1.250 (222 s)
epoch 5: loss 1.222 (278 s)
epoch 6: loss 1.194 (334 s)
epoch 7: loss 1.169 (389 s)
epoch 8: loss 1.152 (445 s)
epoch 9: loss 1.141 (501 s)
epoch 10: loss 1.133 (557 s)
clean 77.2  linf 44.3  l2 62.2  l1 47.8  union 42.4   (69 s)


In [16]:
res_ramp_s0 = {"clean": 79.0, "linf": 42.5, "l2": 62.0, "l1": 45.2, "union": 39.8}

for name, a, b in [("mine", res_mine_s0, res_mine_s1), ("ramp", res_ramp_s0, res_ramp_s1)]:
    mean = {k: (a[k] + b[k]) / 2 for k in a}
    print(name, mean)

NameError: name 'res_mine_s0' is not defined

In [ ]:
# model_mine = adv_train(load_base_model(), my_loss)
# test_model(model_mine)

In [21]:
print(device)
print(torch.cuda.is_available())
!nvidia-smi

cuda
True
Tue Oct 27 09:41:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P0             27W /   70W |    1439MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------------

In [32]:
model_ablate_s1 = adv_train(load_base_model(), my_loss, seed=1, gamma=0.0)
res_ablate_s1 = test_model(model_ablate_s1)

epoch 1: loss 1.431 (52 s)
epoch 2: loss 1.320 (106 s)
epoch 3: loss 1.283 (160 s)
epoch 4: loss 1.248 (215 s)
epoch 5: loss 1.219 (270 s)
epoch 6: loss 1.197 (326 s)
epoch 7: loss 1.167 (381 s)
epoch 8: loss 1.153 (436 s)
epoch 9: loss 1.138 (492 s)
epoch 10: loss 1.138 (548 s)
clean 76.8  linf 45.0  l2 62.1  l1 48.6  union 43.3   (69 s)


In [33]:
model_g5 = adv_train(load_base_model(), my_loss, seed=0, gamma=5.0)
res_g5 = test_model(model_g5)

epoch 1: loss 1.834 (56 s)
epoch 2: loss 1.750 (111 s)
epoch 3: loss 1.623 (167 s)
epoch 4: loss 1.538 (223 s)
epoch 5: loss 1.452 (278 s)
epoch 6: loss 1.386 (334 s)
epoch 7: loss 1.328 (390 s)
epoch 8: loss 1.306 (445 s)
epoch 9: loss 1.286 (501 s)
epoch 10: loss 1.274 (557 s)
clean 77.7  linf 45.7  l2 62.5  l1 46.2  union 42.1   (69 s)


In [18]:
model_mine_s1 = adv_train(load_base_model(), my_loss, seed=1)
res_mine_s1 = test_model(model_mine_s1)
torch.save(model_mine_s1.state_dict(), "mine_s1.pt")

epoch 1: loss 1.502 (56 s)
epoch 2: loss 1.383 (112 s)
epoch 3: loss 1.335 (167 s)
epoch 4: loss 1.296 (223 s)
epoch 5: loss 1.261 (279 s)
epoch 6: loss 1.232 (335 s)
epoch 7: loss 1.199 (391 s)
epoch 8: loss 1.183 (447 s)
epoch 9: loss 1.167 (502 s)
epoch 10: loss 1.168 (558 s)
clean 76.0  linf 45.2  l2 62.1  l1 48.2  union 42.9   (70 s)


In [19]:
res_strong = test_model(model_mine_s1, steps=200, restarts=5)

clean 76.0  linf 44.8  l2 62.0  l1 46.9  union 42.4   (689 s)
